# 01 — Pré-processamento

Transforma os dados brutos em arquivos Parquet limpos e alinha as variáveis macroeconômicas.

## 1. Extração e filtragem: ZIP → Parquet

Descompacta os ZIPs do SCR, filtra apenas registros de **Pessoa Jurídica (PJ)**, converte tipos numéricos e salva um Parquet por ano em `data/processed/scr_pj/`.

In [ ]:
import zipfile
import pandas as pd
from pathlib import Path

BASE = Path().resolve().parent
RAW  = BASE / "data" / "raw" / "scr"
PROC = BASE / "data" / "processed" / "scr_pj"
PROC.mkdir(parents=True, exist_ok=True)

# Colunas quantitativas: no CSV do BCB vem com virgula decimal.
# `numero_de_operacoes` e censurado ("<= 15") por sigilo -> vira NaN em to_numeric.
COLS_NUM = [
    "numero_de_operacoes",
    "a_vencer_ate_90_dias", "a_vencer_de_91_ate_360_dias",
    "a_vencer_de_361_ate_1080_dias", "a_vencer_de_1081_ate_1800_dias",
    "a_vencer_de_1801_ate_5400_dias", "a_vencer_acima_de_5400_dias",
    "vencido_acima_de_15_dias", "carteira_ativa",
    "carteira_inadimplida_arrastada", "ativo_problematico",
]

for ano in range(2012, 2026):
    destino = PROC / f"scr_pj_{ano}.parquet"
    if destino.exists():
        print(f"{ano}: ja processado, pulando")
        continue

    zip_path = RAW / f"scr_{ano}.zip"
    if not zip_path.exists():
        print(f"{ano}: ZIP nao encontrado, pulando")
        continue

    print(f"{ano}: processando...", end=" ", flush=True)
    chunks = []

    with zipfile.ZipFile(zip_path) as z:
        for csv in sorted(f for f in z.namelist() if f.endswith(".csv")):
            # utf-8-sig: os arquivos sao UTF-8 COM BOM. Ler como latin-1 (versao
            # anterior) corrompia os acentos ("Bancario" -> "BancA-rio").
            with z.open(csv) as fh:
                df = pd.read_csv(fh, sep=";", encoding="utf-8-sig", dtype=str)

            df = df[df["cliente"].str.upper().str.contains("PJ", na=False)]   # so PJ
            df["data_base"] = pd.to_datetime(df["data_base"])
            for c in COLS_NUM:
                df[c] = pd.to_numeric(df[c].str.replace(",", ".", regex=False),
                                      errors="coerce")
            chunks.append(df)

    ano_df = pd.concat(chunks, ignore_index=True)
    ano_df.to_parquet(destino, index=False)
    print(f"ok -> {len(ano_df):,} linhas ({destino.stat().st_size/1e6:.1f} MB)")
    del ano_df, chunks   # libera memoria antes do proximo ano

## 2. Inspeção dos arquivos processados

Verifica shape, colunas, valores únicos e nulos de cada parquet gerado.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

PARQ = Path().resolve().parent / "data" / "processed" / "scr_pj"

# Contagem de linhas via metadados do Parquet (nao carrega os dados na memoria)
print("=== Arquivos disponiveis ===")
total = 0
for f in sorted(PARQ.glob("*.parquet")):
    n = pq.ParquetFile(f).metadata.num_rows
    total += n
    print(f"{f.name}: {n:>10,} linhas | {f.stat().st_size/1e6:>6.1f} MB")
print(f"{'TOTAL':<22} {total:>10,} linhas")

# Estrutura: le apenas o ano mais recente
print("\n=== Estrutura (2024) ===")
df = pd.read_parquet(PARQ / "scr_pj_2024.parquet")
print("\n-- Colunas e tipos --")
print(df.dtypes.to_string())
print("\n-- Amostra --")
print(df.head(3).to_string())
print("\n-- Nulos por coluna (top 5) --")
print(df.isnull().mean().sort_values(ascending=False).head(5).to_string())

In [ ]:
import pandas as pd
from pathlib import Path

PROC = Path().resolve().parent / "data" / "processed" / "scr_pj"
df = pd.read_parquet(PROC / "scr_pj_2024.parquet")

print("=== Valores únicos — dimensões ===")
dims = ["uf", "cnae_secao", "porte", "modalidade", "origem", "indexador", "tcb", "sr"]
for col in dims:
    vals = sorted(df[col].dropna().unique())
    print(f"\n{col} ({len(vals)} valores):")
    for v in vals:
        print(f"  {v}")

print("\n=== Meses disponíveis em 2024 ===")
print(sorted(df["data_base"].dt.strftime("%Y-%m").unique()))

print("\n=== Nulos por coluna ===")
print(df.isnull().mean().sort_values(ascending=False))

In [ ]:
import pyarrow.parquet as pq
from pathlib import Path

PARQ = Path().resolve().parent / "data" / "processed" / "scr_pj"

# Confere o schema de cada ano lendo apenas os metadados (nao le os dados)
print("Verificacao de schema (colunas-chave presentes em todos os anos):")
CHAVES = ["data_base", "cnae_secao", "porte", "modalidade",
          "carteira_ativa", "carteira_inadimplida_arrastada"]
for f in sorted(PARQ.glob("*.parquet")):
    cols = set(pq.ParquetFile(f).schema.names)
    faltando = [c for c in CHAVES if c not in cols]
    print(f"  {f.name}: {'OK' if not faltando else 'FALTANDO ' + str(faltando)}"
          f"  ({len(cols)} colunas)")

## 3. Série setorial e consolidação macroeconômica

Primeiro constrói a série de inadimplência dos principais setores — agregando **arquivo a
arquivo**, para não estourar a memória — e em seguida a une às variáveis macroeconômicas,
gerando `dataset_final.parquet`.

> A ordem destas duas células foi corrigida: a versão anterior tentava ler
> `serie_inadimplencia_setorial.parquet` antes de criá-lo, resultando em `FileNotFoundError`
> ao executar o notebook do início ao fim.

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path().resolve().parent
PROC = BASE / "data" / "processed"
PARQ = PROC / "scr_pj"

SETORES = [
    "PJ - Construção",
    "PJ - Comércio; reparação de veículos automotores e motocicletas",
    "PJ - Indústrias de transformação",
    "PJ - Transporte, armazenagem e correio",
    "PJ - Eletricidade e gás",
]
COLUNAS = ["data_base", "cnae_secao", "carteira_ativa", "carteira_inadimplida_arrastada"]

# Agrega ARQUIVO A ARQUIVO: concatenar os 14 anos de uma vez (~95 milhoes de linhas)
# estourava a memoria (ArrowMemoryError). Cada ano vira poucas dezenas de linhas
# agregadas antes de serem unidas.
parciais = []
for f in sorted(PARQ.glob("*.parquet")):
    ano = pd.read_parquet(f, columns=COLUNAS)
    ano = ano[ano["cnae_secao"].isin(SETORES)]
    parciais.append(
        ano.groupby(["data_base", "cnae_secao"], observed=True)
           .agg(carteira=("carteira_ativa", "sum"),
                inadimplida=("carteira_inadimplida_arrastada", "sum"))
           .reset_index()
    )
    del ano

serie = (
    pd.concat(parciais, ignore_index=True)
      .groupby(["data_base", "cnae_secao"], observed=True)[["carteira", "inadimplida"]]
      .sum()
      .reset_index()
      .assign(
          taxa_inadimplencia=lambda x: x["inadimplida"] / x["carteira"] * 100,
          setor=lambda x: x["cnae_secao"].str.replace("PJ - ", "", regex=False).str.strip(),
      )
      .sort_values(["cnae_secao", "data_base"])
)

print(f"Shape: {serie.shape}")
print(f"Periodo: {serie['data_base'].min().date()} -> {serie['data_base'].max().date()}")
print(f"Setores: {serie['setor'].nunique()}  |  meses: {serie['data_base'].nunique()}")

serie.to_parquet(PROC / "serie_inadimplencia_setorial.parquet", index=False)
print("\nSalvo: serie_inadimplencia_setorial.parquet")

In [ ]:
import pandas as pd
from pathlib import Path

PROC = Path().resolve().parent / "data" / "processed"

serie = pd.read_parquet(PROC / "serie_inadimplencia_setorial.parquet")
macro = pd.read_parquet(PROC / "macro.parquet")

# Alinha datas — série setorial é fim de mês, macro pode ser início
macro["data"] = pd.to_datetime(macro["data"]).dt.to_period("M").dt.to_timestamp("M")
serie["data_base"] = pd.to_datetime(serie["data_base"]).dt.to_period("M").dt.to_timestamp("M")

# Join por mês
dataset = serie.merge(macro, left_on="data_base", right_on="data", how="left").drop(columns="data")

# Interpola nulos pontuais (PIB trimestral, PNAD mensal com defasagem)
colunas_macro = [c for c in macro.columns if c != "data"]
dataset[colunas_macro] = (
    dataset.sort_values("data_base")[colunas_macro]
    .interpolate(method="linear")
)

print(f"Shape final: {dataset.shape}")
print(f"Período: {dataset['data_base'].min()} → {dataset['data_base'].max()}")
print(f"\nNulos restantes:")
print(dataset.isnull().sum())

dataset.to_parquet(PROC / "dataset_final.parquet", index=False)
print("\nSalvo em dataset_final.parquet")